# Phase 3 · Feature Engineering & Selection
### Trích xuất đặc trưng có cơ sở bảo mật, phân tích đa cộng tuyến và chọn lọc theo tác động lên biến mục tiêu

| 5W1H | Nội dung |
|---|---|
| **What** | (1) Kiểm kê ba họ đặc trưng: **20 đặc trưng cũ**, **60 đặc trưng thủ công mới + TLD**, **TF-IDF n-gram**. (2) Phân tích tương quan và **VIF**. (3) Xếp hạng tác động bằng **Mutual Information, ANOVA F / χ², Tree importance, Permutation importance**. (4) Chốt tập đặc trưng cho Phase 4. |
| **Why** | Phase 1 cho thấy 14 bộ mô tả đơn giản để lại 8.7% URL mơ hồ, và code cũ có đặc trưng chết, đếm sai từ khóa. Đặc trưng **dư thừa** gây hại cho mô hình tuyến tính (hệ số không ổn định), còn đặc trưng **nhiễu** làm tăng chi phí và nguy cơ overfit. |
| **Where** | Đầu vào: `artifacts/data/*` của Phase 2. Đầu ra: `artifacts/data/{split}_legacy.parquet`, `artifacts/transformers/tfidf_chi2_selector.joblib`, `artifacts/reports/phase3_*.{csv,json}`. |
| **How** | Mọi thống kê dùng để **chọn** đặc trưng được tính trên **Train**. Validation chỉ dùng để **xác nhận** rằng tập rút gọn không làm giảm hiệu năng. **Test không được chạm tới.** |

In [ ]:
import sys
import json
import time
import inspect
import warnings
from pathlib import Path


def _find_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data' / 'raw' / 'phishing_site_urls.csv').exists():
            return p
    raise FileNotFoundError('Không tìm thấy thư mục gốc dự án.')


ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from scipy import sparse, stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

from sklearn.feature_selection import mutual_info_classif, f_classif, chi2, SelectKBest
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import average_precision_score

from src.phishing_ml.config import DATA_DIR, TRANSFORMER_DIR, REPORT_DIR, RANDOM_STATE, seed_everything
from src.phishing_ml.features import (FEATURE_GROUPS, FEATURE_TO_GROUP, LEGACY_COLS, build_feature_frame,
                                      legacy_feature_frame, row_level_features)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
seed_everything()
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
pd.set_option('display.max_columns', 40)
SPLITS = ['train', 'val', 'test']
STAT_SAMPLE = 100_000          # mẫu train cho MI / tương quan / VIF

meta = json.loads((REPORT_DIR / 'phase2_metadata.json').read_text())
NUM_COLS, BINARY_COLS = meta['num_cols'], meta['binary_cols']
CONT_COLS = [c for c in NUM_COLS if c not in BINARY_COLS]
R = {k: pd.read_parquet(DATA_DIR / f'{k}_raw.parquet') for k in SPLITS}        # giá trị gốc
S = {k: pd.read_parquet(DATA_DIR / f'{k}_scaled.parquet') for k in SPLITS}     # đã chuẩn hóa
T = {k: sparse.load_npz(DATA_DIR / f'{k}_tfidf.npz') for k in SPLITS}           # TF-IDF
y = {k: R[k]['label'].to_numpy() for k in SPLITS}
ALL_FEATURES = NUM_COLS + ['tld_te']
# Ma trận cho mô hình cây: giá trị gốc + TLD đã target-encode (cây bất biến với scaling)
X_tree = {k: R[k][NUM_COLS].astype(np.float32).assign(tld_te=S[k]['tld_te'].to_numpy()) for k in SPLITS}
print({k: X_tree[k].shape for k in SPLITS}, '| TF-IDF:', {k: T[k].shape for k in SPLITS})

## 1. Feature Extraction: ba họ đặc trưng

### 1.1. Kế thừa 20 đặc trưng CŨ (notebook 04/05)

Các hàm trong `src/features/lexical_features.py` và `host_features.py` được gọi **nguyên vẹn, không sửa**, để Phase 4 tái lập baseline cũ trên cùng split. Lưu ý các đặc trưng cũ được tính trên **URL thô** (không chuẩn hóa), đúng như code cũ.

In [ ]:
t0 = time.time()
L = {k: legacy_feature_frame(R[k]['url']) for k in SPLITS}
for k in SPLITS:
    L[k].to_parquet(DATA_DIR / f'{k}_legacy.parquet', index=False)
print(f'Trích xuất 20 đặc trưng cũ cho {sum(len(v) for v in L.values()):,} URL trong {time.time() - t0:.1f}s')

legacy_audit = pd.DataFrame({
    'Số giá trị khác nhau (train)': L['train'].nunique(),
    'Variance (train)': L['train'].var(),
    'Spearman ρ với nhãn': L['train'].corrwith(pd.Series(y['train']), method='spearman'),
})
legacy_audit['Trạng thái'] = np.where(legacy_audit['Số giá trị khác nhau (train)'] <= 1, '❌ ĐẶC TRƯNG CHẾT (hằng số)', 'OK')
display(legacy_audit)

**Các khiếm khuyết của bộ đặc trưng cũ được minh họa bằng ví dụ đối kháng:**

In [ ]:
probe = ['gobankingrates.com/banking/checking-accounts/',      # 'banking' hợp lệ trong domain
         'en.wikipedia.org/wiki/Pineapple',                       # 'apple' nằm trong 'pineapple'
         'accounts.google.com/signin',                            # brand chính chủ
         'paypal.com.secure-login.verify-account.xyz/webscr.php', # brand giả mạo ngoài domain
         '3232235777/login.php',                                  # IP dạng số nguyên
         'ikmbtfomlqp.deliverdarken.ru/start_page.exe']           # DGA + malware
old = legacy_feature_frame(probe)[['count_sensitive_words', 'has_ip_address_in_hostname', 'has_suspicious_tld']]
new = build_feature_frame(probe)[['sens_in_path', 'sens_in_host', 'brand_out_of_domain', 'sld_is_brand',
                                   'host_is_ip', 'tld', 'sld_entropy', 'ext_executable']]
display(pd.concat({'CŨ': old, 'MỚI': new}, axis=1).set_index(pd.Index(probe, name='URL')))

**Nhận xét: vì sao bộ đặc trưng cũ phải được thay thế**

- **Đặc trưng chết:** `get_domain_age_days` và `get_domain_lifespan_days` là hằng số −1 (WHOIS bị tắt). `has_https` gần như chết: chỉ một số rất nhỏ URL có scheme, nên variance ≈ 0 và ρ = 0.008.
- **Khớp chuỗi con sai:** `count_sensitive_words` đếm `pineapple` = 1 (vì chứa `apple`), `gobankingrates.com/...` = 2, còn `accounts.google.com/signin` (trang chính chủ) = 3, **cao hơn** nhiều URL phishing. Hàm không phân biệt brand nằm *trong domain chính chủ* hay *ở chỗ giả mạo*.
- **Bỏ sót kỹ thuật né tránh:** IP dạng số nguyên `3232235777` có `has_ip_address_in_hostname = 0`. URL DGA kèm file `.exe` có cả 3 đặc trưng cũ đều bằng 0, tức **vô hình** với mô hình cũ.
- **Bộ mới xử lý đúng từng trường hợp:** `pineapple` → 0 từ khóa; `accounts.google.com` → `sld_is_brand = 1` (tín hiệu an toàn); URL giả mạo PayPal → `brand_out_of_domain = 1`, `sens_in_host = 4`, TLD `xyz`; IP số nguyên → `host_is_ip = 1`; URL DGA → TLD `ru` + `ext_executable = 1`.

### 1.2. Bộ đặc trưng thủ công MỚI (kế thừa notebook 06): 60 biến số + 1 biến phân loại

Mỗi nhóm nhắm tới **một kỹ thuật tấn công cụ thể**. Đây là khác biệt cốt lõi so với bộ cũ (chỉ đếm ký tự). Đặc trưng được sinh bằng **registry tường minh** (`FEATURE_GROUPS`), nên thêm một hàm phụ trợ không làm thay đổi schema mô hình (lỗi auto-discovery cũ đã nhặt nhầm `ensure_scheme`).

In [ ]:
catalog = pd.DataFrame([{'Nhóm': g, 'Số đặc trưng': len(fs), 'Kỹ thuật tấn công mục tiêu': atk,
                         'Đặc trưng': ', '.join(fs)} for g, (fs, atk) in FEATURE_GROUPS.items()])
display(catalog)
print(f'Tổng: {catalog["Số đặc trưng"].sum()} đặc trưng (60 số + tld). '
      f'Bị loại ở Phase 2 do hằng số trên train: {meta["constant_cols_dropped"] or "không có"}')
print('\nTrích đoạn hàm trích xuất (src/phishing_ml/features.py → row_level_features):')
print(''.join(inspect.getsource(row_level_features).splitlines(True)[:14]) + '    ...')

**Bằng chứng định lượng: biểu diễn mới chứa nhiều thông tin hơn.** Ta đo tỉ lệ URL train mà vector đặc trưng **trùng hệt** với một URL khác nhãn. Đây là vùng **lỗi Bayes không thể giảm**, vì mọi mô hình buộc phải đoán sai ít nhất một bên.

In [ ]:
def ambiguity_rate(frame: pd.DataFrame, target) -> float:
    key = pd.util.hash_pandas_object(frame.round(6), index=False)
    return float(pd.Series(target).groupby(key.to_numpy()).transform('nunique').gt(1).mean())


amb = pd.DataFrame({
    'Số đặc trưng': [L['train'].shape[1], X_tree['train'].shape[1]],
    '% vector trùng lặp (bất kể nhãn)': [L['train'].duplicated(keep=False).mean() * 100,
                                         X_tree['train'].duplicated(keep=False).mean() * 100],
    '% URL mơ hồ (vector trùng, khác nhãn)': [ambiguity_rate(L['train'], y['train']) * 100,
                                              ambiguity_rate(X_tree['train'], y['train']) * 100],
}, index=['20 đặc trưng CŨ (URL thô)', '61 đặc trưng MỚI (URL chuẩn hóa)'])
display(amb)

**Kết quả:** với 20 đặc trưng cũ, **62.0%** URL train có vector trùng với một URL khác, và **20.98%** URL rơi vào vùng mơ hồ (trùng vector nhưng khác nhãn). Với 61 đặc trưng mới, hai tỉ lệ này giảm còn **6.86%** và **0.23%**, tức **vùng lỗi Bayes không thể giảm thu hẹp khoảng 90 lần**. Đây là bằng chứng định lượng rằng nút thắt của phiên bản cũ nằm ở **biểu diễn**, không nằm ở thuật toán.

### 1.3. Đặc trưng N-gram (TF-IDF char 3–5 + token)

Ma trận TF-IDF đã được fit trên Train ở Phase 2. Ở đây ta xem **những n-gram nào mang tín hiệu mạnh nhất** theo thống kê χ² (tính trên Train). χ² đo mức độ phụ thuộc giữa tần suất n-gram và nhãn, và phù hợp với dữ liệu không âm như TF-IDF.

In [ ]:
char_vec = joblib.load(TRANSFORMER_DIR / 'tfidf_char.joblib')
word_vec = joblib.load(TRANSFORMER_DIR / 'tfidf_word.joblib')
VOCAB = np.concatenate([char_vec.get_feature_names_out().astype(str),
                        np.char.add('tok:', word_vec.get_feature_names_out().astype(str))])

t0 = time.time()
chi2_all, chi2_p = chi2(T['train'], y['train'])
chi2_all = np.nan_to_num(chi2_all)
pos_mean = np.asarray(T['train'][y['train'] == 1].mean(axis=0)).ravel()
neg_mean = np.asarray(T['train'][y['train'] == 0].mean(axis=0)).ravel()
print(f'χ² cho {T["train"].shape[1]:,} n-gram: {time.time() - t0:.1f}s | '
      f'{(chi2_p < 0.05 / len(chi2_p)).sum():,} n-gram có ý nghĩa sau hiệu chỉnh Bonferroni')

order = np.argsort(-chi2_all)
top_ph = [i for i in order if pos_mean[i] > neg_mean[i]][:20]
top_lg = [i for i in order if pos_mean[i] < neg_mean[i]][:20]
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
axes[0].barh([f"'{VOCAB[i]}'" for i in top_ph][::-1], chi2_all[top_ph][::-1], color='#e34a33')
axes[0].set_title('Top 20 n-gram liên quan PHISHING (χ², train)', fontweight='bold')
axes[1].barh([f"'{VOCAB[i]}'" for i in top_lg][::-1], chi2_all[top_lg][::-1], color='#2b8cbe')
axes[1].set_title('Top 20 n-gram liên quan HỢP LỆ (χ², train)', fontweight='bold')
for ax in axes:
    ax.set_xlabel('χ²')
plt.tight_layout()
plt.show()

**Nhận xét: n-gram mang tín hiệu gì?** Có 9,263 n-gram có ý nghĩa sau hiệu chỉnh Bonferroni.
- **Phía phishing:** `login`, `paypal`, `webscr`, `cmd`, `secure`, `admin`, `dispatch` mô phỏng luồng đăng nhập hoặc thanh toán. `wp`, `includes`, `images`, `js`, `php` là thư mục của **CMS WordPress/Joomla bị chiếm quyền**: kẻ tấn công không sở hữu domain, chúng chỉ cấy trang giả vào thư mục của site hợp lệ. `battle` liên quan tới các chiến dịch giả mạo Battle.net. `ru` và `br` là TLD rủi ro. `www` nằm trong **path hoặc subdomain** (vì `www.` ở đầu host đã bị chuẩn hóa) là dấu hiệu nhúng domain giả như `/www.paypal.com/`.
- **Phía hợp lệ:** các nền tảng lớn (`facebook`, `myspace`, `imdb`, `linkedin`, `yahoo`, `wiki`), các TLD đáng tin (`org`, `edu`) và các từ ngôn ngữ tự nhiên (`the`, `of`, `people`, `name`).
- ⚠️ **Artefact thời gian:** token `2011` nghiêng về hợp lệ, vì nó phản ánh **thời điểm thu thập** dữ liệu hợp lệ chứ không phải tính chất độc hại. Đây là hạn chế của dataset cần ghi nhận khi triển khai thực tế.

## 2. Phân tích tương quan (Correlation Heatmap)

Ta dùng **Spearman ρ** vì phần lớn biến lệch nặng và quan hệ giữa chúng là đơn điệu nhưng không tuyến tính. Thứ tự hàng và cột được **sắp theo phân cụm phân cấp** (hierarchical clustering trên khoảng cách 1 − |ρ|), nhờ vậy các **khối đặc trưng đồng tuyến** hiện ra thành các ô vuông sẫm màu dọc đường chéo.

In [ ]:
rs = np.random.default_rng(RANDOM_STATE)
samp = rs.choice(len(y['train']), STAT_SAMPLE, replace=False)
Xs_tree, ys = X_tree['train'].iloc[samp], y['train'][samp]
corr = Xs_tree.corr(method='spearman').fillna(0)

link = hierarchy.average(squareform(1 - corr.abs().to_numpy(), checks=False))
ordered = corr.index[hierarchy.leaves_list(link)]
fig, ax = plt.subplots(figsize=(24, 20))
sns.heatmap(corr.loc[ordered, ordered], cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, linewidths=0.2,
            cbar_kws={'shrink': 0.6, 'label': 'Spearman ρ'}, ax=ax)
ax.set_title('Ma trận tương quan Spearman (train, sắp theo phân cụm phân cấp)', fontsize=17, fontweight='bold')
plt.tight_layout()
plt.show()

upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
high_pairs = (upper.stack().rename('ρ').reset_index().rename(columns={'level_0': 'Đặc trưng 1', 'level_1': 'Đặc trưng 2'})
              .loc[lambda d: d['ρ'].abs() >= 0.8].sort_values('ρ', key=np.abs, ascending=False).reset_index(drop=True))
print(f'{len(high_pairs)} cặp có |ρ| ≥ 0.8 (trong đó {(high_pairs["ρ"].abs() >= 0.9).sum()} cặp ≥ 0.9):')
display(high_pairs)

**Nhận xét: cấu trúc tương quan**

Phân cụm phân cấp làm lộ ra **6 khối đồng tuyến** dọc đường chéo, mỗi khối đo cùng một khái niệm:

| Khối | Đặc trưng | ρ tiêu biểu | Nguyên nhân |
|---|---|:-:|---|
| Query string | `query_len`, `n_params`, `n_equals` | 0.95 – 0.998 | Mỗi tham số có một dấu `=` |
| Độ dài / entropy | `url_len`, `n_tokens`, `url_entropy`, `path_entropy`, `path_len`, `path_depth`, `n_slashes` | 0.84 – 0.97 | URL dài thì path dài và nhiều token |
| Subdomain | `host_n_dots`, `subdomain_levels`, `subdomain_len` | 0.88 – 0.99 | Mỗi cấp subdomain thêm một dấu chấm |
| SLD / host | `sld_len`, `sld_entropy`, `host_len`, `host_entropy` | 0.83 – 0.89 | Chuỗi dài thì entropy cao |
| Thành phần ký tự | `digit_ratio` ↔ `letter_ratio`, `special_ratio` ↔ `avg_token_len` | −0.90 | Các tỉ lệ có tổng bằng 1 |
| Thương hiệu | `sld_is_brand` ↔ `typosquat_dist`, `brand_in_path` ↔ `brand_out_of_domain` | −0.97 / 0.82 | Khoảng cách bằng 0 khi và chỉ khi SLD là brand |

Phần còn lại của ma trận (góc trên trái: các cờ né tránh và payload như `ext_*`, `is_shortener`, `has_port`) gần như **trắng hoàn toàn**. Các nhóm đặc trưng bảo mật mới vì vậy mang **thông tin độc lập** với nhóm cấu trúc, đúng mục tiêu thiết kế.

## 3. Đa cộng tuyến: Variance Inflation Factor (VIF)

Tương quan cặp chỉ phát hiện quan hệ **hai biến**. **VIF** phát hiện cả trường hợp một biến là **tổ hợp tuyến tính của nhiều biến khác**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

trong đó $R_j^2$ là hệ số xác định khi hồi quy biến j theo **tất cả** biến còn lại. Thay vì chạy 61 hồi quy, ta dùng đẳng thức tương đương **VIF = đường chéo của nghịch đảo ma trận tương quan**, áp dụng trên dữ liệu **đã chuẩn hóa** (Phase 2). Ngưỡng diễn giải: VIF < 5 là chấp nhận được, 5–10 là trung bình, > 10 là **nghiêm trọng**.

In [ ]:
def compute_vif(frame: pd.DataFrame) -> pd.Series:
    r = np.corrcoef(frame.to_numpy(dtype=np.float64), rowvar=False)
    return pd.Series(np.diag(np.linalg.pinv(r)), index=frame.columns)


Xs_scaled = S['train'].iloc[samp][ALL_FEATURES]
vif0 = compute_vif(Xs_scaled).sort_values(ascending=False)
vif_tbl = vif0.to_frame('VIF').assign(
    Mức=lambda d: np.select([d['VIF'] > 1e4, d['VIF'] > 10, d['VIF'] > 5], ['Hoàn hảo (tổ hợp tuyến tính)', 'Nghiêm trọng',
                                                                           'Trung bình'], 'Chấp nhận'),
    Nhóm=lambda d: d.index.map(lambda f: FEATURE_TO_GROUP.get(f, 'TLD (phân loại)')))
print(vif_tbl['Mức'].value_counts().to_dict())
display(vif_tbl.head(20).style.format({'VIF': '{:,.1f}'}))

fig, ax = plt.subplots(figsize=(18, 5))
v = vif0.clip(upper=1e3)
ax.bar(v.index, v.values, color=np.where(vif0 > 10, '#e34a33', np.where(vif0 > 5, '#fdae61', '#1b9e77')))
ax.axhline(10, ls='--', c='k', lw=1, label='VIF = 10')
ax.axhline(5, ls=':', c='grey', lw=1, label='VIF = 5')
ax.set_yscale('log')
ax.set(title='VIF từng đặc trưng (train, đã chuẩn hóa; cắt ở 1,000 để hiển thị)', ylabel='VIF (log)')
ax.tick_params(axis='x', rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

**Nhận xét: đa cộng tuyến nghiêm trọng nhưng tập trung cục bộ**

- Có **24/61 biến VIF > 10** (nghiêm trọng) và 7 biến ở mức trung bình. Max VIF lên tới **khoảng 8,000** ở nhóm query string: sau Yeo-Johnson, `query_len`, `n_params` và `n_equals` gần như là **cùng một biến**. Các khối nghiêm trọng khác là độ dài/entropy, subdomain, và các tỉ lệ ký tự (có tổng bằng 1).
- **Tác động:** với mô hình tuyến tính, ma trận XᵀX gần suy biến nên hệ số có phương sai rất lớn. Hai biến gần như trùng nhau có thể nhận hệ số +50 và −49 mà vẫn cho cùng dự đoán, làm mất khả năng diễn giải. Với mô hình cây, đa cộng tuyến **không** làm sai dự đoán, chỉ làm feature importance bị **chia sẻ** giữa các biến trong cùng khối.
- **Xử lý:** xem §5. Ta lọc VIF lặp cho nhánh tuyến tính và **giữ nguyên** cho nhánh cây.

## 4. Tác động lên biến mục tiêu (Target Impact)

Mỗi phương pháp có thiên lệch riêng, nên ta dùng **bốn góc nhìn** và chỉ tin vào kết luận **nhất quán giữa các phương pháp**:

| Phương pháp | Đo cái gì | Điểm mạnh | Điểm yếu |
|---|---|---|---|
| **Mutual Information** | Phụ thuộc thống kê bất kỳ (kể cả phi tuyến) giữa X và Y | Không giả định dạng quan hệ | Đơn biến, ước lượng kNN có nhiễu |
| **ANOVA F (biến liên tục) / χ² (biến nhị phân)**, quy về **r²** | Mức chênh lệch trung bình (F) hoặc tần suất (χ²) giữa hai lớp | Có p-value, dễ diễn giải | Chỉ bắt quan hệ tuyến tính hoặc đơn điệu, đơn biến |
| **LightGBM gain** | Tổng mức giảm loss khi chia theo biến | Đa biến, có tương tác | Thiên vị biến liên tục, nhiều giá trị |
| **Permutation importance** (trên **Validation**) | Mức giảm PR-AUC khi xáo trộn biến | Đo đóng góp **thực tế**, ngoài mẫu | Bị "chia sẻ" giữa các biến đồng tuyến |

Để so sánh được giữa biến liên tục và biến nhị phân, ta quy cả F và χ² về **r²**: với 2 lớp, η² = F/(F + n − 2) chính là bình phương tương quan point-biserial, còn φ² = χ²/n là bình phương tương quan giữa hai biến nhị phân.

In [ ]:
# (a) Mutual Information
t0 = time.time()
mi = pd.Series(mutual_info_classif(Xs_tree, ys, discrete_features=[c in BINARY_COLS for c in ALL_FEATURES],
                                   n_neighbors=5, random_state=RANDOM_STATE), index=ALL_FEATURES)
print(f'Mutual Information: {time.time() - t0:.1f}s')

# (b) ANOVA F cho biến liên tục (dùng dữ liệu đã chuẩn hóa) / χ² cho biến nhị phân
n_tr = len(y['train'])
cont_feats = CONT_COLS + ['tld_te']
F_stat, F_p = f_classif(S['train'][cont_feats], y['train'])
C_stat, C_p = chi2(R['train'][BINARY_COLS], y['train'])
stat_tbl = pd.concat([
    pd.DataFrame({'Kiểm định': 'ANOVA F', 'Thống kê': F_stat, 'p-value': F_p, 'r²': F_stat / (F_stat + n_tr - 2)},
                 index=cont_feats),
    pd.DataFrame({'Kiểm định': 'χ²', 'Thống kê': C_stat, 'p-value': C_p, 'r²': C_stat / n_tr}, index=BINARY_COLS),
])
ALPHA_BONF = 0.05 / len(ALL_FEATURES)
stat_tbl['Có ý nghĩa (Bonferroni)'] = stat_tbl['p-value'] < ALPHA_BONF

# (c) LightGBM gain (fit trên train, early stopping trên val)
t0 = time.time()
lgbm_probe = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                                subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                scale_pos_weight=meta['scale_pos_weight'], random_state=RANDOM_STATE, verbose=-1)
lgbm_probe.fit(X_tree['train'], y['train'], eval_set=[(X_tree['val'], y['val'])], eval_metric='average_precision',
               callbacks=[lgb.early_stopping(50, verbose=False)])
gain = pd.Series(lgbm_probe.booster_.feature_importance('gain'), index=ALL_FEATURES)
gain_share = gain / gain.sum()
print(f'LightGBM probe: {lgbm_probe.best_iteration_} cây, val PR-AUC = '
      f'{average_precision_score(y["val"], lgbm_probe.predict_proba(X_tree["val"])[:, 1]):.4f} ({time.time() - t0:.1f}s)')

# (d) Permutation importance trên Validation
t0 = time.time()
pv = rs.choice(len(y['val']), 30_000, replace=False)
perm = permutation_importance(lgbm_probe, X_tree['val'].iloc[pv], y['val'][pv], scoring='average_precision',
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
perm_mean = pd.Series(perm.importances_mean, index=ALL_FEATURES)
perm_std = pd.Series(perm.importances_std, index=ALL_FEATURES)
print(f'Permutation importance: {time.time() - t0:.1f}s')

In [ ]:
impact = pd.DataFrame({
    'Nhóm': [FEATURE_TO_GROUP.get(f, 'TLD (phân loại)') for f in ALL_FEATURES],
    'MI': mi, 'r² (F/χ²)': stat_tbl['r²'], 'p-value': stat_tbl['p-value'],
    'LGBM gain (%)': gain_share * 100, 'Permutation ΔPR-AUC': perm_mean, 'Perm std': perm_std,
}, index=ALL_FEATURES)
RANK_COLS = ['MI', 'r² (F/χ²)', 'LGBM gain (%)', 'Permutation ΔPR-AUC']
ranks = impact[RANK_COLS].rank(pct=True)
impact['Consensus score'] = ranks.mean(axis=1)
impact = impact.sort_values('Consensus score', ascending=False)
impact['Hạng'] = np.arange(1, len(impact) + 1)
impact.to_csv(REPORT_DIR / 'phase3_feature_impact.csv')
display(impact.style.format({'MI': '{:.4f}', 'r² (F/χ²)': '{:.4f}', 'p-value': '{:.1e}', 'LGBM gain (%)': '{:.2f}',
                             'Permutation ΔPR-AUC': '{:+.4f}', 'Perm std': '{:.4f}', 'Consensus score': '{:.3f}'})
        .background_gradient(subset=['Consensus score'], cmap='Greens'))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(26, 11), sharey=False)
for ax, col, color in zip(axes, RANK_COLS, ['#7570b3', '#d95f02', '#1b9e77', '#e7298a']):
    top = impact[col].sort_values().tail(25)
    ax.barh(top.index, top.values, color=color,
            xerr=impact.loc[top.index, 'Perm std'] if col == 'Permutation ΔPR-AUC' else None)
    ax.set_title(f'Top 25: {col}', fontweight='bold')
plt.suptitle('Bốn góc nhìn về tác động của đặc trưng lên nhãn Phishing', fontsize=16, y=1.0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 15))
sns.heatmap(ranks.loc[impact.index], cmap='viridis', cbar_kws={'label': 'Percentile rank'}, ax=ax)
ax.set_title('Percentile rank theo từng phương pháp (sắp theo Consensus)', fontweight='bold')
plt.tight_layout()
plt.show()

**Nhận xét: những gì các phương pháp đồng thuận và không đồng thuận**

1. **Đồng thuận gần như tuyệt đối ở hai vị trí đầu:** `tld_te` (rủi ro của TLD) đứng **#1 ở cả bốn phương pháp**. `sens_in_path` (từ khóa lừa đảo trong path) đứng #2 ở ba phương pháp và #3 theo MI (sau `tld_len`). Hai đặc trưng này chiếm 37% tổng gain của LightGBM, và xáo trộn chúng làm PR-AUC giảm lần lượt 0.163 và 0.117. Logic bảo mật: kẻ tấn công ưu tiên **TLD rẻ, ít kiểm duyệt**, và trang giả phải mô phỏng **luồng đăng nhập** nằm trên hạ tầng của chúng.
2. **Top 10 consensus** gồm `tld_te`, `sens_in_path`, `longest_token` (hash MD5 32 ký tự), `n_dots` (chuỗi subdomain giả), `brand_out_of_domain` (giả mạo thương hiệu), `digit_ratio`, `host_n_digits` (hạ tầng hosting giá rẻ hoặc DGA), `n_slashes`, `ext_script` (`.php` của phishing kit) và `letter_ratio`. Mỗi đặc trưng tương ứng với một **cơ chế tấn công cụ thể**, không chỉ là một triệu chứng thống kê.
3. **Mức đồng thuận giữa các phương pháp** (Spearman giữa các thang xếp hạng): LightGBM gain ↔ Permutation **0.80**, MI ↔ gain 0.72, nhưng **r² (F/χ²) ↔ Permutation chỉ 0.29**. Kiểm định đơn biến tuyến tính *đánh giá thấp* những biến có tác động phi tuyến. Ví dụ `longest_token` chỉ có r² = 0.029 nhưng đứng #3 consensus, còn `path_entropy` có r² = 0.003 nhưng permutation vẫn dương rõ. Đây là **xác nhận độc lập** cho kết luận phi tuyến của Phase 1, và là lý do không được chọn đặc trưng chỉ bằng F-test.
4. **Cuối bảng** là các cờ cực hiếm trong train (`has_userinfo` có 9 URL, `host_non_ascii` 59, `host_has_punycode` 64) và các cờ **trùng thông tin** với một biến mạnh hơn. `host_is_ip` bị TLD `<ip>` hấp thụ (TLD này có target encoding 0.994). `is_typosquat` bị `typosquat_dist` hấp thụ. `brand_in_subdomain` bị `brand_out_of_domain` hấp thụ.
   > 🔐 *Lưu ý bảo mật:* các cờ như `has_userinfo` hay `host_has_punycode` bị loại vì **dataset hiện tại** hầu như không có kỹ thuật tấn công tương ứng, chứ không phải vì kỹ thuật đó vô hại. Code trích xuất vẫn giữ chúng, và nên đánh giá lại khi bổ sung dữ liệu mới có tấn công IDN homograph.

## 5. Lựa chọn tập đặc trưng tối ưu

### 5.1. Quy tắc lựa chọn (được viết thành code, không chọn tay)

**Bước A: Loại đặc trưng nhiễu.** Một đặc trưng bị coi là *không mang thông tin* nếu nó **không đạt ít nhất 3/4 tiêu chí** sau:
1. MI > 0.001 nat
2. Kiểm định F/χ² có ý nghĩa sau hiệu chỉnh Bonferroni (p < 0.05/61)
3. LightGBM gain > 0.1% tổng gain
4. Permutation ΔPR-AUC − 1·std > 0, tức đóng góp ngoài mẫu dương một cách đáng tin cậy

**Bước B: Hai tập cho hai họ mô hình.** Lý do là đa cộng tuyến tác động lên chúng **khác nhau**:
- **`TREE_FEATURES`** = kết quả Bước A. Mô hình cây **không bị** đa cộng tuyến làm sai: ở mỗi nút cây chỉ chọn một trong các biến đồng tuyến. Giữ lại các biến này cho cây thêm lựa chọn điểm chia.
- **`LINEAR_FEATURES`** = `TREE_FEATURES` sau khi **lọc VIF lặp**: chừng nào còn VIF > ngưỡng τ, ta loại biến có **Consensus score thấp nhất** trong số các biến vi phạm, rồi tính lại VIF. Mô hình tuyến tính và MLP **có bị** ảnh hưởng: hệ số không ổn định, phương sai lớn, khó diễn giải.
  Ngưỡng "VIF > 10" trong sách giáo khoa là quy ước **diễn giải hệ số**, chưa chắc tối ưu cho **dự đoán**. Vì vậy ta thử các ngưỡng τ ∈ {∞, 100, 30, 10}. τ = ∞ nghĩa là không lọc và chỉ dựa vào **phạt L2** của Logistic Regression. Phạt L2 (ridge) vốn là cách kinh điển để ổn định hệ số khi có đa cộng tuyến.

**Bước C: Xác nhận trên Validation.**
- Tập cho cây chỉ được chấp nhận nếu PR-AUC trên validation giảm **không quá 0.002** so với tập đầy đủ.
- Tập cho mô hình tuyến tính: chọn tập **ít biến nhất** có PR-AUC nằm trong khoảng 0.002 so với ứng viên tốt nhất (nguyên tắc parsimony).

In [ ]:
criteria = pd.DataFrame({
    'MI > 0.001': impact['MI'] > 0.001,
    'p < α_Bonferroni': impact['p-value'] < ALPHA_BONF,
    'Gain > 0.1%': impact['LGBM gain (%)'] > 0.1,
    'Perm − std > 0': (impact['Permutation ΔPR-AUC'] - impact['Perm std']) > 0,
})
impact['Số tiêu chí đạt'] = criteria.sum(axis=1)
noise = impact.index[impact['Số tiêu chí đạt'] < 3].tolist()
selected_tree = [f for f in impact.index if f not in noise]
print(f'Bước A: loại {len(noise)} đặc trưng nhiễu → còn {len(selected_tree)}/{len(ALL_FEATURES)}')
display(pd.concat([criteria, impact[['Số tiêu chí đạt', 'Consensus score']]], axis=1).loc[noise])

In [ ]:
# Bước B: lọc VIF lặp cho nhánh tuyến tính, với nhiều ngưỡng τ
def vif_prune(features, max_vif):
    log, cur = [], list(features)
    while True:
        vif = compute_vif(S['train'].iloc[samp][cur])
        offenders = vif[vif > max_vif]
        if offenders.empty:
            return cur, log, float(vif.max())
        drop = impact.loc[offenders.index, 'Consensus score'].idxmin()
        log.append({'Vòng': len(log) + 1, 'Loại bỏ': drop, 'VIF lúc loại': vif[drop],
                    'Consensus score': impact.loc[drop, 'Consensus score'], 'Số biến vi phạm': len(offenders),
                    'Max VIF trước': offenders.max()})
        cur.remove(drop)


linear_candidates = {'τ = ∞ (không lọc, dựa vào L2)': (list(selected_tree), [],
                                                       float(compute_vif(S['train'].iloc[samp][selected_tree]).max()))}
for tau in [100, 30, 10]:
    linear_candidates[f'τ = {tau}'] = vif_prune(selected_tree, tau)
for name, (cols, log, mx) in linear_candidates.items():
    print(f'{name:<32} → {len(cols)} biến | loại {len(log):>2} | max VIF = {mx:,.1f}')
print('\nNhật ký lọc với τ = 10 (chặt nhất):')
display(pd.DataFrame(linear_candidates['τ = 10'][1]))

In [ ]:
# Bước C: xác nhận trên Validation
def lgbm_val_score(cols) -> float:
    m = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8,
                           subsample_freq=1, colsample_bytree=0.8, scale_pos_weight=meta['scale_pos_weight'],
                           random_state=RANDOM_STATE, verbose=-1)
    m.fit(X_tree['train'][cols], y['train'], eval_set=[(X_tree['val'][cols], y['val'])],
          eval_metric='average_precision', callbacks=[lgb.early_stopping(50, verbose=False)])
    return average_precision_score(y['val'], m.predict_proba(X_tree['val'][cols])[:, 1])


def lr_val_score(cols) -> float:
    m = LogisticRegression(max_iter=3000, class_weight='balanced').fit(S['train'][cols], y['train'])
    return average_precision_score(y['val'], m.predict_proba(S['val'][cols])[:, 1])


TOL = 0.002
t0 = time.time()
rows = [{'Mô hình kiểm chứng': 'LightGBM', 'Tập': 'Đầy đủ', 'Số biến': len(ALL_FEATURES), 'Max VIF': np.nan,
         'Val PR-AUC': lgbm_val_score(ALL_FEATURES)},
        {'Mô hình kiểm chứng': 'LightGBM', 'Tập': 'Bước A (loại nhiễu)', 'Số biến': len(selected_tree), 'Max VIF': np.nan,
         'Val PR-AUC': lgbm_val_score(selected_tree)},
        {'Mô hình kiểm chứng': 'Logistic Regression (L2)', 'Tập': 'Đầy đủ', 'Số biến': len(ALL_FEATURES),
         'Max VIF': float(vif0.max()), 'Val PR-AUC': lr_val_score(ALL_FEATURES)}]
for name, (cols, _, mx) in linear_candidates.items():
    rows.append({'Mô hình kiểm chứng': 'Logistic Regression (L2)', 'Tập': f'Bước A + VIF {name}', 'Số biến': len(cols),
                 'Max VIF': mx, 'Val PR-AUC': lr_val_score(cols)})
validation = pd.DataFrame(rows)
print(f'Xác nhận trên validation: {time.time() - t0:.1f}s')
display(validation.style.format({'Max VIF': '{:,.1f}', 'Val PR-AUC': '{:.4f}'}, na_rep='—'))

# Cây: chấp nhận tập loại nhiễu nếu không giảm quá TOL
d_tree = validation.iloc[1]['Val PR-AUC'] - validation.iloc[0]['Val PR-AUC']
TREE_FEATURES = selected_tree if d_tree >= -TOL else ALL_FEATURES
# Tuyến tính: tập ÍT BIẾN NHẤT nằm trong TOL so với ứng viên tốt nhất
lin_rows = validation.iloc[3:].reset_index(drop=True)
best_lin = lin_rows['Val PR-AUC'].max()
chosen = lin_rows[lin_rows['Val PR-AUC'] >= best_lin - TOL].sort_values('Số biến').iloc[0]
LINEAR_FEATURES = linear_candidates[chosen['Tập'].replace('Bước A + VIF ', '')][0]
vif_log = linear_candidates[chosen['Tập'].replace('Bước A + VIF ', '')][1]
print(f'TREE_FEATURES  : ΔPR-AUC so với đầy đủ = {d_tree:+.4f} → {len(TREE_FEATURES)} biến')
print(f'LINEAR_FEATURES: chọn "{chosen["Tập"]}" → {len(LINEAR_FEATURES)} biến, max VIF = {chosen["Max VIF"]:,.1f}, '
      f'val PR-AUC = {chosen["Val PR-AUC"]:.4f} (tốt nhất = {best_lin:.4f})')

### 5.2. Chọn lọc N-gram bằng χ² (SelectKBest)

Không gian TF-IDF có khoảng 350 nghìn chiều. Ta thử giữ **k n-gram có χ² cao nhất** (tính trên Train) và đo PR-AUC trên Validation bằng một `SGDClassifier` (log-loss). Mô hình này nhanh, đủ để **so sánh tương đối** giữa các giá trị k. Quy tắc chọn: lấy **k nhỏ nhất** có PR-AUC nằm trong khoảng 0.001 so với k tốt nhất. Ít chiều hơn đồng nghĩa với inference nhanh hơn và mô hình nhỏ hơn.

In [ ]:
k_grid = [10_000, 30_000, 100_000, 200_000, T['train'].shape[1]]
k_rows = []
for k in k_grid:
    t0 = time.time()
    sel = SelectKBest(chi2, k=k).fit(T['train'], y['train'])
    sgd = SGDClassifier(loss='log_loss', alpha=1e-6, max_iter=15, tol=None, class_weight='balanced',
                        random_state=RANDOM_STATE).fit(sel.transform(T['train']), y['train'])
    k_rows.append({'k': k, 'Val PR-AUC': average_precision_score(y['val'], sgd.decision_function(sel.transform(T['val']))),
                   'Thời gian (s)': time.time() - t0})
k_tbl = pd.DataFrame(k_rows)
best_k_score = k_tbl['Val PR-AUC'].max()
TFIDF_K = int(k_tbl.loc[k_tbl['Val PR-AUC'] >= best_k_score - 0.001, 'k'].min())
display(k_tbl)
print(f'→ Chọn k = {TFIDF_K:,} n-gram')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_tbl['k'], k_tbl['Val PR-AUC'], marker='o')
ax.axvline(TFIDF_K, ls='--', c='#e34a33', label=f'k được chọn = {TFIDF_K:,}')
ax.set(xscale='log', xlabel='Số n-gram giữ lại (k, thang log)', ylabel='Validation PR-AUC',
       title='Chọn số n-gram theo χ²')
ax.legend()
plt.tight_layout()
plt.show()

tfidf_selector = SelectKBest(chi2, k=TFIDF_K).fit(T['train'], y['train'])
joblib.dump(tfidf_selector, TRANSFORMER_DIR / 'tfidf_chi2_selector.joblib', compress=3)

## 6. Tổng kết lựa chọn và lưu kết quả

In [ ]:
final_tbl = pd.DataFrame({
    'Đặc trưng': ALL_FEATURES,
    'Nhóm': [FEATURE_TO_GROUP.get(f, 'TLD (phân loại)') for f in ALL_FEATURES],
    'Hạng consensus': impact.loc[ALL_FEATURES, 'Hạng'].to_numpy(),
    'TREE': ['✅' if f in TREE_FEATURES else '—' for f in ALL_FEATURES],
    'LINEAR / MLP': ['✅' if f in LINEAR_FEATURES else '—' for f in ALL_FEATURES],
}).sort_values('Hạng consensus').reset_index(drop=True)
display(final_tbl)

selection = {
    'tree_features': TREE_FEATURES, 'linear_features': LINEAR_FEATURES, 'all_features': ALL_FEATURES,
    'noise_dropped': noise, 'vif_dropped': [r['Loại bỏ'] for r in vif_log], 'tfidf_k': TFIDF_K,
    'legacy_features': LEGACY_COLS,
    'validation': validation.to_dict(orient='records'),
    'top10_consensus': impact.index[:10].tolist(),
}
(REPORT_DIR / 'phase3_feature_selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False))
print(f'Đã lưu: phase3_feature_selection.json, phase3_feature_impact.csv, {{split}}_legacy.parquet, '
      f'tfidf_chi2_selector.joblib')
print(f'TREE = {len(TREE_FEATURES)} | LINEAR = {len(LINEAR_FEATURES)} | TF-IDF k = {TFIDF_K:,}')

### Tóm tắt Phase 3

| Hạng mục | Kết quả | Bằng chứng |
|---|---|---|
| Biểu diễn mới vs cũ | URL mơ hồ **20.98% → 0.23%** | §1.2 |
| Bước A: loại nhiễu | Loại 9 cờ hiếm hoặc trùng thông tin → **52 biến** | LightGBM val PR-AUC **0.9227 → 0.9256** (*tăng* khi bớt biến) |
| Bước B: đa cộng tuyến (nhánh tuyến tính) | VIF τ = 30 loại `subdomain_levels`, `n_params`, `avg_token_len`, `n_tokens` → **48 biến**, max VIF **8,079 → 24.6** | LR val PR-AUC 0.8792 so với tốt nhất 0.8808 (trong khoảng dung sai 0.002) |
| VIF τ = 10 (ngưỡng sách giáo khoa) | 40 biến nhưng PR-AUC **giảm 0.011** | Lọc quá chặt làm mất tín hiệu; ngưỡng 10 dành cho *diễn giải*, không dành cho *dự đoán* |
| N-gram | χ² top-k: chọn **k = 200,000 / 345,923** (bỏ 42% số chiều) | Val PR-AUC 0.9783 so với 0.9785 khi dùng toàn bộ |

**→ Phase 4** dùng: `TREE_FEATURES` (52) cho DT/RF/XGBoost/LightGBM/CatBoost; `LINEAR_FEATURES` (48) cho LR và MLP; TF-IDF 200k chiều cho LR/SVM/NB và Stacking; 20 đặc trưng cũ để tái lập baseline.